<a href="https://colab.research.google.com/github/kinjaljoshi/mech-interp-from-scratch/blob/master/3_sae_metrics_cheatsheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#SAE Feature Analysis: Steps and Practice Drills (GPT-2 small)
A repeatable routine for working with a sparse autoencoder (SAE): computing metrics, selecting features, showing the tokens they fire on, and reading their decoded values. Every step has a short concept, a commented code block, and a drill to redo from scratch until it is automatic. Everything runs on real GPT-2 small activations with a pretrained SAE (no synthetic data).

Shapes to memorize
* Residual activations acts: [batch, pos, d_model] (768 for GPT-2 small)
* Feature activations feats = sae.encode(acts): [batch, pos, d_sae] (24576 for the jb SAE) - 32 * d_model
* sae.W_enc: [d_model, d_sae]; sae.W_dec: [d_sae, d_model]. Row f of W_dec is feature f's direction.
* Reconstruction: x_hat = feats @ W_dec + b_dec

In [ ]:
import torch

x = torch.tensor([[0.0, 3.0, 0.0, 1.0],
                  [2.0, 0.0, 0.0, 5.0]])   # [2 rows, 4 features] like a tiny feats tensor

# topk: biggest k values and their indices along a dimension
val, idx = x.topk(2, dim=-1)               # per row: top 2 values and which columns
print(val)
print(idx)
# boolean mask + sum: count how many entries are "active" (> 0)
fires = (x>0).float().sum(dim=1)    # tensor([2., 2.])
print("fires",fires)
# mean over a dimension: fraction of rows where each column fires
density = (x>0).float().mean(dim=0)       # [4], values in 0..1
print("density",density)
# cosine similarity: normalize rows, then matrix multiply
xn = torch.nn.functional.normalize(x, dim=-1)
cosine = xn @ xn.T
print("cosine",cosine)

# flatten a [batch, pos] grid to find the global top entries
flatten = x.flatten()
value, indices = flatten.topk(3)
print("values",value, "indices", indices)
row, col = indices // x.shape[1], indices % x.shape[1]
print(row, col)
print(x[row,col])
x.shape[1]

tensor([[3., 1.],
        [5., 2.]])
tensor([[1, 3],
        [3, 0]])
fires tensor([2., 2.])
density tensor([0.5000, 0.5000, 0.0000, 1.0000])
cosine tensor([[1.0000, 0.2936],
        [0.2936, 1.0000]])
values tensor([5., 3., 2.]) indices tensor([7, 1, 4])
tensor([1, 0, 1]) tensor([3, 1, 0])
tensor([5., 3., 2.])


4

In [ ]:
!pip install transformer_lens
!pip install sae_lens

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.3 MB/s  0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12526 sha256=16b5fd5cceeb4be1b52fed9ac2a76459493970729df72cec0449a9eca28f417b
  Stored in directory: /root/.cache/pip/wheels/f6/6e/1a/710143d0e50e827ae5b60a47a6d43d715cf1c6e35881a05530
Successfully built transformers-stream-generator
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [transformer_lens]
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for py2store: filename=py2store-0.1.24-py3-none-any.whl size=123427 sha256=8fd18ab1cea87a4c3ba12107a38ca81605f850dd84109c6c9a637a2a7e11060d
  Stored in directory: /root/.cache/pip/wheels/fe/48/10/95af03b77b8f8351058f22f53

In [ ]:
import torch
from transformer_lens import TransformerBridge
from sae_lens import SAE

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TransformerBridge.boot_transformers("gpt2", device=device)
TransformerBridge.enable_compatibility_mode(model)
hook_name = "blocks.8.hook_resid_pre"

sae = SAE.from_pretrained(release="gpt2-small-res-jb", sae_id=hook_name, device = device)

if isinstance(sae, tuple):
    sae = sae[0]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

... (5 outputs left out)

blocks.8.hook_resid_pre/sae_weights.safe(…): downloading bytes:           |  0.00B            

blocks.8.hook_resid_pre/sparsity.safeten(…): reconstructing file:   0%|          |  0.00B / 98.4kB            

blocks.8.hook_resid_pre/sparsity.safeten(…): downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/sae_lens/saes/sae.py:256: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = SAETransformerBridge.boot_transformers(..., **cfg.metadata.model_from_pretrained_kwargs)
or, with transformer-lens<4.0:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.metadata.model_from_pretrained_kwargs)
  warnings.warn(


In [ ]:
texts = [
    "The Eiffel Tower is in Paris, and the Louvre is a museum in Paris too.",
    "def add(a, b):\n    return a + b",
    "She said that the stock market fell by 3% on Tuesday.",
    "The cat sat on the mat. The dog sat on the log.",
    "Dr. Smith prescribed 20 mg of ibuprofen twice a day.",
    "import numpy as np\nx = np.zeros((3, 3))",
    "In 1969, Apollo 11 landed on the Moon.",
    "I love how the sunset looks over the ocean tonight.",
]

tokens = model.to_tokens(texts)
print(f"tokens shape - {tokens.shape}")
str_tokens = [model.to_str_tokens(t) for t in tokens]
print(f"str_tokens shape - {len(str_tokens)}")

logits, cache = model.run_with_cache(tokens, names_filter=hook_name)
acts = cache[hook_name]
print(f"acts shape - {acts.shape}")
feats = sae.encode(acts)
print(f"feats shape - {feats.shape}")

recon = sae.decode(feats)
print(f"recon shape - {recon.shape}")

tokens shape - torch.Size([8, 21])
str_tokens shape - 8
acts shape - torch.Size([8, 21, 768])
feats shape - torch.Size([8, 21, 24576])
recon shape - torch.Size([8, 21, 768])


In [ ]:
# mask: real tokens only (not BOS at pos 0, not padding)
lengths = torch.tensor([len(s) for s in str_tokens])
print("shape of lengths", lengths)
print(f"tokens shape - {tokens.shape}")
print(torch.arange(tokens.shape[1]).shape)
pos_idx = torch.arange(tokens.shape[1])[None, :]
print(f"pos_idx shape {pos_idx.shape}")
print(f"lengths[:, None] {lengths[:, None]}")
mask = (pos_idx >0) & (pos_idx < lengths[:, None])
print("mask shape", mask.shape, mask)

shape of lengths tensor([21, 21, 21, 21, 21, 21, 21, 21])
tokens shape - torch.Size([8, 21])
torch.Size([21])
pos_idx shape torch.Size([1, 21])
lengths[:, None] tensor([[21],
        [21],
        [21],
        [21],
        [21],
        [21],
        [21],
        [21]])
mask shape torch.Size([8, 21]) tensor([[False,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True],
        [False,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True],
        [False,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True],
        [False,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          Tr

In [ ]:
m = mask

x, xh, f = acts[m], recon[m], feats[m] # boolean indexing flattens to [n_tokens, ...]
print(f"activation shape - {x.shape}")
print(f"reconstruction shape - {xh.shape}")
print(f"feature shape - {f.shape}")

activation shape - torch.Size([160, 768])
reconstruction shape - torch.Size([160, 768])
feature shape - torch.Size([160, 24576])


## Evaluating Sparse Autoencoder (SAE) Health: The 4 Core Metrics

Four core metrics tell you whether an SAE is learning clean, useful representations: **L0 sparsity**, **reconstruction quality**, **feature firing health**, and **downstream loss recovered**.

---

### 1. Sparsity ($L_0$ Norm)
Measures the average number of active (non-zero) features firing simultaneously per token.

$$L_0 = \frac{1}{N} \sum_{i=1}^{N} \|z_i\|_0 = \frac{1}{N} \sum_{i=1}^{N} \sum_{j=1}^{M} \mathbb{I}(z_{i,j} > 0)$$

* **$z_i$:** Hidden feature activation vector for token $i$ (out of $M$ total dictionary features).
* **Interpretation:** Lower is sparser, while higher means more features fire together. Healthy SAEs typically target **$L_0 \approx 20\text{–}100$** depending on model size.

---

### 2. Reconstruction Quality (MSE, FVU, and Cosine Similarity)
Measures how accurately the SAE's output $\hat{x} = \text{decode}(\text{encode}(x))$ reproduces the original activation $x$.

* **Mean Squared Error (MSE):** Average squared Euclidean distance between original and reconstructed activations.
  $$\text{MSE} = \frac{1}{N} \sum_{i=1}^{N} \|x_i - \hat{x}_i\|_2^2$$

* **Fraction of Variance Unexplained (FVU):** Scale-free error normalized by total dataset variance (equivalent to $1 - R^2$).
  $$\text{FVU} = \frac{\sum_{i=1}^{N} \|x_i - \hat{x}_i\|_2^2}{\sum_{i=1}^{N} \|x_i - \bar{x}\|_2^2}$$
  *(Target: **$< 0.10$** is strong; **$1.0$** means the SAE is no better than predicting the constant dataset mean $\bar{x}$.)*

* **Cosine Similarity:** Measures directional alignment independent of vector magnitude.
  $$\text{Cosine}(x, \hat{x}) = \frac{1}{N} \sum_{i=1}^{N} \frac{x_i \cdot \hat{x}_i}{\|x_i\|_2 \|\hat{x}_i\|_2}$$
  *(Target: **$> 0.95$** indicates high directional fidelity.)*

---

### 3. Feature Health (Dead vs. Dense Features)
Tracks the firing frequency $f_j$ of each individual feature $j$ across a validation dataset of $N$ tokens.

$$f_j = \frac{1}{N} \sum_{i=1}^{N} \mathbb{I}(z_{i,j} > 0)$$

* **Dead Features ($f_j = 0$ or $< 10^{-5}$):** Neurons that never (or almost never) fire, representing wasted dictionary capacity.
* **Healthy / Specific Features ($0.0001 \le f_j \le 0.05$):** Fire selectively on meaningful concepts, syntax, or entities.
* **Ultra-Dense Features ($f_j > 0.10$):** Fire too frequently across unrelated tokens, often acting as generic bias terms or low-level punctuation/position detectors.

---

### 4. Downstream Loss Recovered (CE Loss Score)
Measures how much of the language model's next-token prediction ability survives when true activations at `hook_name` are replaced with SAE reconstructions.

$$\text{Loss Recovered} = \frac{\mathcal{L}_{\text{zeroed}} - \mathcal{L}_{\text{spliced}}}{\mathcal{L}_{\text{zeroed}} - \mathcal{L}_{\text{clean}}}$$

* **$\mathcal{L}_{\text{clean}}$:** Cross-entropy loss (in nats) of the untouched model.
* **$\mathcal{L}_{\text{spliced}}$:** Loss when activations are replaced with $\hat{x} = \text{sae.decode}(\text{sae.encode}(x))$.
* **$\mathcal{L}_{\text{zeroed}}$:** Worst-case baseline loss when activations are replaced with zeros.
* **Interpretation:** **$1.0$** means zero downstream performance loss (strong SAEs score **$0.90\text{–}0.99$**); **$0.0$** means no better than zero ablation.

---

### Quick Summary Reference

| Metric | What It Checks | Healthy Target | Failure Warning |
| :--- | :--- | :--- | :--- |
| **$L_0$** | Active features per token | $20 - 100$ | Too high = dense/polysemantic; too low = underfitting |
| **FVU** | Unexplained variance ($1 - R^2$) | $< 0.10$ | $\ge 1.0$ = worse than predicting the mean |
| **Cosine Sim** | Directional alignment | $> 0.95$ | $< 0.85$ = poor geometric reconstruction |
| **Dead Features** | Unused dictionary neurons | $< 5\% - 10\%$ | $> 30\%$ = severe capacity waste |
| **Loss Recovered** | Downstream model behavior | $0.90 - 0.99$ | $< 0.80$ = critical causal signal lost |


In [ ]:
#l0 norm
l0 = (f>0).float().sum(dim=-1).mean()
print(f"l0 shape {l0.shape} l0 norm -# avg active features per token = {l0}")

# 1. Firing rate per feature (fraction of tokens where each feature > 0)
fire_rate = (f > 0).float().mean(dim=0)

# 2. Strictly dead features (never fired once in the entire dataset)
dead_mask = (fire_rate == 0)
dead_indices = torch.where(dead_mask)
print(f"Dead features: {dead_mask.sum().item()} / {f.shape} ({dead_mask.float().mean():.1%})")
# avg squared error per token
mse = ((x - xh) ** 2).sum(dim=-1).mean()
print(f"mse - avg squared error per token = {mse}")
# fraction of variance unexplained; explained var = 1 - fvu
fvu = ((x - xh) ** 2).sum() / ((x - x.mean(0)) ** 2).sum()
print(f"fvu - fraction of variance unexplained = {fvu}")

cos = torch.nn.functional.cosine_similarity(x, xh, dim=-1)

print(f"cos - cosine similarity = {cos.mean()}")

l0 shape torch.Size([]) l0 norm -# avg active features per token = 45.243751525878906
Dead features: 21348 / torch.Size([160, 24576]) (86.9%)
mse - avg squared error per token = 726.033935546875
fvu - fraction of variance unexplained = 0.11407686769962311
cos - cosine similarity = 0.9682912826538086


### Evaluating Downstream Impact: Loss Recovered

While **FVU** measures geometric reconstruction error at a single layer, **Loss Recovered** (CE Loss Score) tests whether the SAE preserves the functional information the rest of the language model needs to predict next tokens accurately.

#### 1. Intervention Hooks
Using TransformerLens hooks, we intercept the residual stream at `hook_name` and compare three forward passes on the same `tokens` batch:
* **`clean` (Best-Case Ceiling):** Normal forward pass with untouched activations (lowest loss).
* **`spliced` (SAE Reconstruction):** Replaces the true activations `a` with `sae.decode(sae.encode(a))` and lets the rest of the model run downstream.
* **`zeroed` (Worst-Case Baseline):** Replaces activations with zeros (`torch.zeros_like(a)`) to measure how badly the model degrades when all information at that layer is wiped out.

#### 2. Formula & Interpretation
$$\text{Loss Recovered} = \frac{\text{Loss}_{\text{zeroed}} - \text{Loss}_{\text{spliced}}}{\text{Loss}_{\text{zeroed}} - \text{Loss}_{\text{clean}}}$$

* **`1.0` (100%):** Perfect functional reconstruction—the model predicts just as well with SAE activations as with original activations (strong SAEs typically score **0.90 – 0.99**).
* **`0.0` (0%):** The SAE reconstruction is no better than zeroing out the layer completely.
* **`< 0.0`:** The SAE introduces reconstruction noise that hurts downstream predictions worse than zero ablation.


In [ ]:
# Loss recovered: how much of the model's loss survives when the SAE replaces the residual
def splice_rep(a, hook):   return sae.decode(sae.encode(a))     # replace activations with SAE reconstruction
def zero_ablation(a, hook): return torch.zeros_like(a)           # worst-case baseline

clean   = model(tokens, return_type="loss")
spliced = model.run_with_hooks(tokens, return_type="loss", fwd_hooks=[(hook_name, splice_rep)])
zeroed  = model.run_with_hooks(tokens, return_type="loss", fwd_hooks=[(hook_name, zero_ablation)])
loss_recovered = (zeroed - spliced) / (zeroed - clean)       # 1.0 = perfect, 0.0 = as bad as zeroing
print(f"loss clean={clean:.3f} loss spliced={spliced:.3f} loss zeroed={zeroed:.3f} loss recovered={loss_recovered:.3f}")

loss clean=5.771 loss spliced=5.978 loss zeroed=11.497 loss recovered=0.964


Concept: Each feature gets its own statistics: how often it fires (density), how hard (max and mean-when-active), and whether it is dead (never fires) or dense (fires on a large share of tokens, usually uninteresting).

In [ ]:
# A. Top by max activation (strongest, often most specific)
f = feats[mask] # [n_features, d_sae]
print(f"[n_features, d_sae] {f.shape}")

density = (f>0).float().mean(dim=0)
print(f"[d_sae] fraction of tokens each feature fires on {density.shape}")

#firing count
n_active = (f>0).sum(dim=0)
print(f"[d_sae] firing counts {n_active.shape}")

#never fired on THIS sample (not necessarily dead globally)
dead = (n_active == 0)
print(f"never fired on THIS sample (not necessarily dead globally) {dead.shape}")

dense = density >0.1
print(f"dead on this sample: {dead.float().mean():.1%}, dense: {dense.sum().item()} features")


mean_when_active = f.sum(0) / n_active.clamp(min=1)
print(f"mean when active {mean_when_active.shape}")
max_when_active = f.max(0).values
print(f"max when active {max_when_active.shape}")


[n_features, d_sae] torch.Size([160, 24576])
[d_sae] fraction of tokens each feature fires on torch.Size([24576])
[d_sae] firing counts torch.Size([24576])
never fired on THIS sample (not necessarily dead globally) torch.Size([24576])
dead on this sample: 86.9%, dense: 28 features
mean when active torch.Size([24576])
max when active torch.Size([24576])


In [ ]:
# A. Top by max activation (strongest, often most specific)
top_max = max_when_active.topk(10)
print(top_max.indices.tolist(), top_max.values.tolist())

# B. Top by density (most common; usually syntax or position features)
top_dense = density.topk(10).indices
print(f"top dense - {top_dense}")

# C. Features that fire on one specific token (e.g. every " Paris")
target = " Paris"
sel = torch.zeros_like(mask)
for b, toks in enumerate(str_tokens):
  for p, t in enumerate(toks):
    if t == target:
            sel[b, p] = True
feat_on_token = feats[sel]

[20644, 12435, 9303, 326, 19941, 6965, 69, 348, 2445, 1763] [73.997802734375, 73.11119079589844, 73.01049041748047, 66.25453186035156, 62.27344512939453, 57.554832458496094, 56.50627899169922, 55.775184631347656, 55.194942474365234, 54.645477294921875]
top dense - tensor([ 6863,  7678, 18486, 20644, 19683, 23092, 14287, 16021,  5926, 15075],
       device='cuda:0')


In [ ]:
mean_on_feats = feat_on_token.mean(dim=0)
print(f"mean_on_feature = shape {mean_on_feats.shape} value = {mean_on_feats}")
top_for_token = mean_on_feats.topk(10)
print(top_for_token.indices.tolist(), top_for_token.values.tolist())

mean_on_feature = shape torch.Size([24576]) value = tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0',
       grad_fn=<MeanBackward1>)
[974, 18593, 7329, 12354, 7533, 17551, 19692, 2281, 7314, 24462] [42.0289192199707, 9.795580863952637, 7.970775604248047, 6.576241493225098, 5.491601467132568, 5.357389450073242, 4.936460494995117, 4.886824607849121, 4.226419925689697, 3.1057276725769043]


### Differential Feature Discovery (Contrastive Analysis)

To find SAE features specialized for a specific domain (e.g., **code vs. prose**), we compare average feature activations across two contrasting sets of prompts.

#### 1. Step-by-Step Breakdown
* **Group Selection (`idx_a`, `idx_b`):** Hardcoded row indices pointing to specific prompt categories in our test batch (e.g., `idx_a = ` for code prompts vs. `idx_b = ` for prose prompts).
* **Valid Token Masking (`feats[idx][mask[idx]]`):** Applies a boolean mask to remove padding and BOS tokens, flattening the remaining real tokens into shape `[num_valid_tokens, num_features]`.
* **Group Means (`mean_a`, `mean_b`):** Averages each feature's activation across all valid tokens in that group (`shape: [num_features]`).
* **Contrastive Ranking (`diff = mean_a - mean_b`):** Subtracts baseline prose activity from code activity. Calling `.topk(10)` surfaces the 10 features most selective for code over prose.

#### 2. Formula
$$\Delta_j = \frac{1}{|T_A|} \sum_{t \in T_A} z_{t,j} - \frac{1}{|T_B|} \sum_{t \in T_B} z_{t,j}$$

* **$T_A, T_B$:** Sets of valid (non-padded) tokens in Prompt Group A and Prompt Group B.
* **$z_{t,j}$:** Activation of feature $j$ on token $t$.

#### 3. Practical Caveat: Magnitude vs. Consistency
Ranking by raw activation difference ($\Delta_j$) can favor features that spike intensely on a single keyword (e.g., `def` or `import`). To find features that fire **consistently** across an entire domain, compare **firing frequency differences** instead:

$$\Delta_{\text{freq}, j} = \text{Mean}\big(\mathbb{I}(z_{T_A, j} > 0)\big) - \text{Mean}\big(\mathbb{I}(z_{T_B, j} > 0)\big)$$


In [ ]:
#differential analysis code versus prose

# D. Differential: features that fire more on prompt set A than prompt set B
idx_a, idx_b = [1, 5], [0, 2, 3, 6]                       # code texts vs. prose texts
mean_a = feats[idx_a][mask[idx_a]].mean(0)
mean_b = feats[idx_b][mask[idx_b]].mean(0)
diff = mean_a - mean_b
print(f"diff shape = {diff.shape}")
top_code_features = diff.topk(10)
print(top_code_features.indices.tolist(), top_code_features.values.tolist())

prose_selective_features = (-diff).topk(10)
print(prose_selective_features.indices.tolist(), prose_selective_features.values.tolist())

diff shape = torch.Size([24576])
[9303, 14190, 10958, 7323, 24198, 22336, 13321, 6965, 13031, 17084] [4.248326301574707, 4.13990592956543, 2.514618396759033, 2.3342368602752686, 2.284275770187378, 2.1884002685546875, 1.9007538557052612, 1.819567084312439, 1.76823890209198, 1.7609840631484985]
[20644, 7678, 18486, 6863, 974, 7920, 6069, 12435, 1756, 19404] [3.8916845321655273, 2.3027968406677246, 1.495913028717041, 1.1405059099197388, 1.0507229566574097, 0.9937669634819031, 0.956899106502533, 0.9138898849487305, 0.8356105089187622, 0.8113991022109985]


In [ ]:
def top_contexts(feature, k=5, window=6):
    """Print the k strongest activations of one feature with surrounding tokens."""
    vals = feats[:, :, feature].clone()                   # [batch, pos]
    vals[~mask] = -1.0                                    # exclude BOS and padding
    top = vals.flatten().topk(min(k, int(mask.sum())))
    for v, flat_i in zip(top.values, top.indices):
        b, p = divmod(int(flat_i), vals.shape[1])         # flat index back to (text, position)
        toks = str_tokens[b]
        left  = "".join(toks[max(0, p - window):p])
        right = "".join(toks[p + 1:p + 1 + window])
        print(f"{v:6.2f} | {left!r} [[{toks[p]}]] {right!r}")

def feature_on_prompt(feature, text_idx):
    """Token-by-token activation of one feature on one prompt."""
    for t, a in zip(str_tokens[text_idx], feats[text_idx, :len(str_tokens[text_idx]), feature]):
        bar = "#" * int(a.item() * 2)
        print(f"{t!r:>16} {a.item():6.2f} {bar}")

fid = int(top_max.indices[0])
top_contexts(fid)
feature_on_prompt(fid, 0)

 74.00 | '3, 3))<|endoftext|><|endoftext|>' [[<|endoftext|>]] ''
 72.63 | '((3, 3))<|endoftext|>' [[<|endoftext|>]] '<|endoftext|>'
 71.37 | ' a + b<|endoftext|><|endoftext|><|endoftext|>' [[<|endoftext|>]] '<|endoftext|>'
 71.15 | ' return a + b<|endoftext|><|endoftext|>' [[<|endoftext|>]] '<|endoftext|><|endoftext|>'
 71.09 | ' + b<|endoftext|><|endoftext|><|endoftext|><|endoftext|>' [[<|endoftext|>]] ''
 '<|endoftext|>'   0.00 
           'The'   0.00 
            ' E'   0.00 
           'iff'   0.00 
            'el'   0.00 
        ' Tower'   0.00 
           ' is'   0.00 
           ' in'   0.00 
        ' Paris'   0.00 
             ','   0.00 
          ' and'   0.00 
          ' the'   0.00 
          ' Lou'   0.00 
           'vre'   0.00 
           ' is'   0.00 
            ' a'   0.00 
       ' museum'   0.00 
           ' in'   0.00 
        ' Paris'   0.00 
          ' too'   0.00 
             '.'   0.00 


Step 6: Decoded values (what the feature writes)
Concept: The decoder row W_dec[f] is the direction the feature adds to the residual stream. Reading it two ways gives you its meaning: through the unembedding (what next-token it promotes) and through similarity to other decoder rows (feature neighbors, possible splitting or absorption).

### Direct Logit Attribution (Logit Lens on SAE Features)

While inspecting top-activating text snippets shows **what triggers a feature** (input side), projecting its decoder vector onto the unembedding matrix $W_U$ shows **what next tokens the feature directly promotes or suppresses** (output side).

#### 1. Formula
$$\text{logit\_effect} = W_{\text{dec}}[j] \cdot W_U$$

* **$W_{\text{dec}}[j]$ (`shape: [d_model]`):** The decoder weight vector for feature $j$—the exact direction added back into the residual stream when the feature fires.
* **$W_U$ (`shape: [d_model, d_vocab]`):** The model's unembedding matrix that maps residual stream directions to vocabulary logits.
* **Result (`shape: [d_vocab]`):**
  * **Large positive values:** Vocabulary tokens whose output probability is **directly boosted** when feature $j$ activates.
  * **Large negative values:** Vocabulary tokens whose output probability is **directly suppressed**.

#### 2. Key Approximation Caveats
Projecting $W_{\text{dec}}[j]$ directly through $W_U$ provides a fast linear approximation that skips two downstream components:
1. **Intermediate Layers:** Subsequent attention heads and MLPs may read, transform, or cancel out this direction before the final layer.
2. **Final LayerNorm (`ln_final`):** Skips the final residual stream normalization, which rescales overall logit magnitudes (though the relative ranking of top tokens remains largely intact).


In [ ]:
Wd = sae.W_dec                                            # [d_sae, d_model]

# (a) Direct logit effect of feature fid (approximation: skips later layers and final layernorm)
logit_effect = Wd[fid] @ model.W_U

In [ ]:
# [d_vocab]
top = logit_effect.topk(10)
bot = (-logit_effect).topk(10)
print("promotes:", model.to_str_tokens(top.indices))
print("suppresses:", model.to_str_tokens(bot.indices))

promotes: ['Untitled', 'Abstract', 'Description', 'WASHINGTON', 'CLOSE', 'Overview', 'CTV', 'SAN', 'Welcome', 'Still']
suppresses: [' theirs', ' circulation', ' embargo', ' reper', ' amnesty', ' displacement', ' obligation', ' established', ' abandon', ' limit']


### Nearest Decoder Neighbors (Cosine Similarity)

To check whether a feature represents a unique concept or has split into redundant duplicates across the dictionary, we find its closest neighbors in the SAE decoder weight space ($W_{\text{dec}}$).

#### 1. Formula
$$\text{sim}(j, k) = \frac{W_{\text{dec}}[j] \cdot W_{\text{dec}}[k]}{\|W_{\text{dec}}[j]\|_2 \, \|W_{\text{dec}}[k]\|_2}$$

* **Row Normalization (`Wn`):** Each decoder vector is scaled to unit length ($\|w\|_2 = 1$), allowing pairwise cosine similarity to be computed via a single matrix-vector dot product (`Wn @ Wn[fid]`).
* **Top-6 Selection:** Calling `.topk(6)` returns the feature itself at rank 1 (`cos = 1.000`) followed by its **5 closest neighboring features**.

#### 2. Interpretation
* **`0.10 – 0.35` (Healthy Separation):** Distinct features sitting in a related semantic neighborhood (e.g., a *Python functions* feature near a *Python imports* feature).
* **`> 0.70` (Feature Splitting / Redundancy):** Multiple features point in nearly the same direction, splitting a single concept across duplicate neurons.
* **`≈ 0.00` (Orthogonal / Isolated):** The feature shares almost no directional overlap with other dictionary features.


In [ ]:
# (b) Nearest decoder neighbors by cosine similarity
Wn = torch.nn.functional.normalize(Wd, dim=-1)
sims = Wn @ Wn[fid]                                       # [d_sae]
nn = sims.topk(6)                                         # first entry is the feature itself (cos = 1)
print(list(zip(nn.indices.tolist(), [round(v, 3) for v in nn.values.tolist()])))


[(20644, 1.0), (7678, 0.947), (21490, 0.933), (18486, 0.92), (6955, 0.623), (818, 0.623)]


In [ ]:
# (c) A feature's actual contribution to the reconstruction at one token
b, p = 0, 5
contrib = feats[b, p, fid] * Wd[fid]                      # [d_model] = activation * decoder direction
frac = contrib.norm() / recon[b, p].norm()                # how large it is relative to the whole reconstruction
print(f"feature {fid} contributes {frac:.2%} of recon norm at token {str_tokens[b][p]!r}")

feature 20644 contributes 0.00% of recon norm at token ' Tower'


In [ ]:
# (d) Decompose one token: its top features and how much each adds
vals, idx = feats[b, p].topk(8)
for v, i in zip(vals, idx):
    print(int(i), round(float(v), 2), round(float((v * Wd[i]).norm()), 2))

1066 36.61 36.61
16031 7.72 7.72
22529 7.35 7.35
3117 6.66 6.66
9873 6.15 6.15
7307 5.68 5.68
3794 5.47 5.48
23417 5.33 5.33


/tmp/ipykernel_12465/934377198.py:4: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  print(int(i), round(float(v), 2), round(float((v * Wd[i]).norm()), 2))


In [ ]:
def make_ablation_hook(feature):
    def hook(a, hook):
        f = sae.encode(a)                       # [batch, pos, d_sae]
        err = a - sae.decode(f)                 # keep the SAE error term so only the feature changes
        f[:, :, feature] = 0.0                  # zero this feature everywhere
        return sae.decode(f) + err
    return hook

prompt = "The Eiffel Tower is in the city of"
toks = model.to_tokens(prompt)
base = model(toks)[0, -1]
abl  = model.run_with_hooks(toks, fwd_hooks=[(hook_name, make_ablation_hook(fid))])[0, -1]
paris = model.to_single_token(" Paris")
print("Paris logit change:", (abl[paris] - base[paris]).item())

Paris logit change: 1.9073486328125e-06
